# Interactive Dashboards

Interactive dashboards let users ask follow-up questions without a new report: **filters** narrow the data, **actions** connect charts (click a bar to filter the others), **drill-down** moves from a summary to details, and **parameters** change what is computed. BI tools offer these as menu options; this lesson builds the same patterns in Python, so you understand what they do.

**After this lesson you can:**

1. Add global filters and parameters to a dashboard.
2. Connect charts with filter and highlight actions.
3. Build drill-down from borough to zone.
4. Decide which interactions help and which only add complexity.

**Data:** NYC yellow taxi trips, January 2024, aggregated by zone, day and hour.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance",
        "fare_amount", "tip_amount", "total_amount", "payment_type"]

def load_month(month: str) -> pd.DataFrame:
    """One month of NYC yellow taxi trips with the cleaning rules of M08 Lesson 08."""
    df = pd.read_parquet(datasets.taxi(month), columns=COLS)
    start = pd.Timestamp(month + "-01")
    dur = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    ok = (df["tpep_pickup_datetime"].between(start, start + pd.offsets.MonthBegin(1), inclusive="left")
          & dur.between(1, 180) & df["trip_distance"].between(0.1, 100) & df["fare_amount"].between(3, 500))
    return df[ok].assign(duration_min=dur[ok])

zones = pd.read_csv(datasets.fetch("taxi_zone_lookup.csv"))

import altair as alt
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"
trips = load_month("2024-01")
agg = (trips.assign(date=trips["tpep_pickup_datetime"].dt.normalize(), hour=trips["tpep_pickup_datetime"].dt.hour)
       .groupby(["PULocationID", "date", "hour"]).agg(trips=("fare_amount", "size"), revenue=("total_amount", "sum")).reset_index()
       .merge(zones.rename(columns={"LocationID": "PULocationID"}), on="PULocationID"))
agg = agg[agg["Borough"].isin(["Manhattan", "Queens", "Brooklyn", "Bronx"])]
by_bdh = agg.groupby(["Borough", "date", "hour"], as_index=False)[["trips", "revenue"]].sum()
print(f"{len(agg):,} zone-day-hour rows; {len(by_bdh):,} borough-day-hour rows for the charts")

72,094 zone-day-hour rows; 2,864 borough-day-hour rows for the charts


## 1. Filter Action: Click a Bar to Filter the Other Charts

Click a borough in the bar chart: the daily line and the hour profile show only that borough. Click on the empty background to clear the selection. Shift-click selects several boroughs.

In [2]:
alt.data_transformers.disable_max_rows()
pick = alt.selection_point(fields=["Borough"], empty=True)
color = alt.condition(pick, alt.Color("Borough:N", legend=None, scale=alt.Scale(scheme="tableau10")), alt.value("#DDDDDD"))

bars = alt.Chart(by_bdh).mark_bar().encode(
    y=alt.Y("Borough:N", sort="-x", title=None), x=alt.X("sum(trips):Q", title="Trips"), color=color,
    tooltip=[alt.Tooltip("sum(trips):Q", format=",", title="Trips")],
).add_params(pick).properties(width=260, height=160, title="Click a borough")

line = alt.Chart(by_bdh).mark_line(point=True).encode(
    x=alt.X("yearmonthdate(date):T", title=None), y=alt.Y("sum(trips):Q", title="Trips per day"),
).transform_filter(pick).properties(width=480, height=160, title="Daily trips (filtered)")

profile = alt.Chart(by_bdh).mark_area(opacity=0.8).encode(
    x=alt.X("hour:O", title="Hour"), y=alt.Y("sum(trips):Q", title="Trips"),
).transform_filter(pick).properties(width=760, height=150, title="Trips by hour (filtered)")

(bars | line) & profile

alt.VConcatChart(...)

## 2. Parameters: Let the User Choose the Measure

A **parameter** is a user input that changes a calculation. Here a drop-down switches the measure between trips, revenue and revenue per trip; the same chart updates.

In [3]:
measure = alt.param(name="measure", value="trips", bind=alt.binding_select(options=["trips", "revenue", "revenue_per_trip"],
                                                                          name="Measure: "))
by_bh = by_bdh.groupby(["Borough", "hour"], as_index=False)[["trips", "revenue"]].sum()
by_bh["revenue_per_trip"] = by_bh["revenue"] / by_bh["trips"]
long = by_bh.melt(id_vars=["Borough", "hour"], value_vars=["trips", "revenue", "revenue_per_trip"], var_name="m")
alt.Chart(long).mark_line(point=True).encode(
    x=alt.X("hour:O", title="Hour"), y=alt.Y("value:Q", title="Selected measure"),
    color=alt.Color("Borough:N", scale=alt.Scale(scheme="tableau10")),
    tooltip=["Borough", "hour", alt.Tooltip("value:Q", format=",.2f")],
).add_params(measure).transform_filter("datum.m == measure").properties(width=640, height=280,
                                                                         title="Parameter-driven chart: choose the measure")

alt.Chart(...)

## 3. Drill-Down: Borough → Zone

Drill-down lets users start with a summary and open details on demand. A **treemap** with a hierarchy supports it natively: click a borough to zoom into its zones, click the header to go back up.

In [4]:
zone_tot = agg.groupby(["Borough", "Zone"], as_index=False).agg(trips=("trips", "sum"), revenue=("revenue", "sum"))
zone_tot["revenue_per_trip"] = zone_tot["revenue"] / zone_tot["trips"]
fig = px.treemap(zone_tot, path=[px.Constant("NYC"), "Borough", "Zone"], values="trips", color="revenue_per_trip",
                 color_continuous_scale="viridis", hover_data={"revenue_per_trip": ":.2f"},
                 title="Trips by borough and zone (size) and revenue per trip (colour) - click to drill down")
fig.update_layout(height=550, margin=dict(t=50, l=10, r=10, b=10))
fig.show()

## 4. Highlight vs Filter, and the Cost of Interaction

| Interaction | Effect | Use when |
| :--- | :--- | :--- |
| filter | removes other data | the user wants to focus on a subset |
| highlight | keeps context, dims the rest | comparison with the whole matters |
| drill-down | changes the level of detail | hierarchical data (region → store) |
| parameter | changes the calculation | alternative measures, thresholds, top N |
| tooltip | shows details on demand | values the chart cannot label |

Each interaction adds something the user must discover and remember. Many dashboard users never click, so the **default view** must answer the main question on its own.

The cell checks the performance side: how much data each chart embeds in the page, a common cause of slow dashboards.

In [5]:
import json

specs = {"linked bars/line/profile": ((bars | line) & profile).to_dict(),
         "treemap": json.loads(fig.to_json())}
for name, spec in specs.items():
    print(f"{name:<26} {len(json.dumps(spec)) / 1024:8.0f} KiB of JSON in the page")
raw_rows = len(trips)
print(f"\nraw trips: {raw_rows:,} -> chart rows: {len(by_bdh):,} (aggregation is what keeps the page small)")

linked bars/line/profile        285 KiB of JSON in the page
treemap                          34 KiB of JSON in the page

raw trips: 2,855,769 -> chart rows: 2,864 (aggregation is what keeps the page small)


*Insight: Add interaction where it answers the next natural question (which borough? which zone? which hour?), keep the default view self-explanatory, and pre-aggregate data so the dashboard stays fast.*

## Real-World Use Case or Analogy

An interactive dashboard is like a **museum guide app**: the main hall gives the overview; tapping an exhibit opens details; a filter shows only the rooms of one period. If the app needs a manual, visitors close it and look at the walls instead.

In Tableau these features are called *filters*, *dashboard actions*, *hierarchies* and *parameters*; in Power BI *slicers*, *cross-filtering*, *drill-down* and *field parameters*. The ideas and the design rules are the same.